In [1]:
import pyomo.environ as pyo

n = 7
I = range(1, n+1)

c = {
    (1,1):999, (1,2):12,  (1,3):10,   (1,4):999, (1,5):999, (1,6):999,(1,7):12,
    (2,1):12,  (2,2):999, (2,3):8,   (2,4):12,  (2,5):999, (2,6):999, (2,7):999,
    (3,1):10,  (3,2):8,   (3,3):999, (3,4):11,  (3,5):3,  (3,6):999,  (3,7):9,
    (4,1):999, (4,2):12,  (4,3):11,  (4,4):999, (4,5):11, (4,6):10,  (4,7):999,
    (5,1):999, (5,2):999, (5,3):3,   (5,4):11,  (5,5):999, (5,6):6,   (5,7):7,
    (6,1):999, (6,2):999, (6,3):999, (6,4):10,  (6,5):6,   (6,6):999, (6,7):9,
    (7,1):12, (7,2):999,  (7,3):9,   (7,4):999, (7,5):7,   (7,6):9,  (7,7):999
}

model = pyo.ConcreteModel()

model.I = pyo.Set(initialize=I)
model.J = pyo.Set(initialize=I)

model.x = pyo.Var(model.I, model.J, within=pyo.Binary)
model.u = pyo.Var(model.I, within=pyo.NonNegativeReals)
model.z = pyo.Var()

def obj_rule(m):
    return m.z == sum(m.x[i,j] * c[i,j] for i in m.I for j in m.J)
model.obj = pyo.Constraint(rule=obj_rule)
model.obj_func = pyo.Objective(expr=model.z, sense=pyo.minimize)


# r1(i): sum_j x(i,j) = 1
def r1_rule(m, i):
    return sum(m.x[i,j] for j in m.J) == 1
model.r1 = pyo.Constraint(model.I, rule=r1_rule)

# r2(j): sum_i x(i,j) = 1
def r2_rule(m, j):
    return sum(m.x[i,j] for i in m.I) == 1
model.r2 = pyo.Constraint(model.J, rule=r2_rule)

# r3(i,j): MTZ subtours (solo para i>1, j>1 y i != j)
def r3_rule(m, i, j):
    if i > 1 and j > 1 and i != j:
        return m.u[i] - m.u[j] + n * m.x[i,j] <= n - 1
    return pyo.Constraint.Skip
model.r3 = pyo.Constraint(model.I, model.J, rule=r3_rule)

model.u_fix = pyo.Constraint(expr=model.u[1] == 0)

model.u_bounds = pyo.ConstraintList()
for i in I:
    if i == 1:
        model.u_bounds.add(model.u[1] == 0)
    else:
        model.u_bounds.add(model.u[i] >= 1)
        model.u_bounds.add(model.u[i] <= n-1)

model.no_loops = pyo.ConstraintList()
for i in I:
    model.no_loops.add(model.x[i,i] == 0)


solver = pyo.SolverFactory('glpk')
solver.options['mipgap'] = 0.0001

result = solver.solve(model, tee=True)


print("Costo óptimo =", pyo.value(model.z))

selected = [(i,j) for i in I for j in I if pyo.value(model.x[i,j]) > 0.5]
print("\nRutas seleccionadas (aristas):")
for i,j in selected:
    print(f"{i} -> {j}")

succ = {i: None for i in I}
for i,j in selected:
    succ[i] = j

tour = [1]
current = 1
visited = set([1])
while True:
    nxt = succ.get(current, None)
    if nxt is None:
        print("No hay sucesor para", current)
        break
    tour.append(nxt)
    if nxt == 1:
        break
    if nxt in visited:
        break
    visited.add(nxt)
    current = nxt

print("\nTour reconstruido (orden):")
print(" -> ".join(map(str,tour)))


GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --mipgap 0.0001 --write C:\Users\Vivi\AppData\Local\Temp\tmpoxe603m9.glpk.raw
 --wglp C:\Users\Vivi\AppData\Local\Temp\tmpkjpecxlm.glpk.glp --cpxlp C:\Users\Vivi\AppData\Local\Temp\tmpzahlxb9w.pyomo.lp
Reading problem data from 'C:\Users\Vivi\AppData\Local\Temp\tmpzahlxb9w.pyomo.lp'...
C:\Users\Vivi\AppData\Local\Temp\tmpzahlxb9w.pyomo.lp:525: warning: lower bound of variable 'x4' redefined
C:\Users\Vivi\AppData\Local\Temp\tmpzahlxb9w.pyomo.lp:525: warning: upper bound of variable 'x4' redefined
66 rows, 57 columns, 259 non-zeros
49 integer variables, all of which are binary
574 lines were read
Writing problem data to 'C:\Users\Vivi\AppData\Local\Temp\tmpkjpecxlm.glpk.glp'...
450 lines were written
GLPK Integer Optimizer, v4.65
66 rows, 57 columns, 259 non-zeros
49 integer variables, all of which are binary
Preprocessing...
30 constraint coefficient(s) were reduced
44 rows, 48 columns, 174 non-zeros
42 integ

# Heurístico

Algoritmo de Inserción
- Inicialización
    Seleccionar un ciclo inicial (subtour) con k vértices.
    Hacer W = V \ {vértices seleccionados}.
- Mientras (W ≠ ∅ )
    Tomar j de W de acuerdo con algún criterio preestablecido
    Insertar j donde menos incremente la longitud del ciclo
    Hacer W = W \ {j}.

In [2]:
import numpy as np

INF = float('inf')

dist = np.array([
    [0, 12, 10, INF, INF, INF, 12],
    [12, 0, 8, 12, INF, INF, INF],
    [10, 8, 0, 11, 3, INF, 9],
    [INF, 12, 11, 0, 11, 10, INF],
    [INF, INF, 3, 11, 0, 6, 7],
    [INF, INF, INF, 10, 6, 0, 9],
    [12, INF, 9, INF, 7, 9, 0]
])

cycle = [1, 2, 3, 1]  
W = set(range(1, 8)) - set(cycle[:-1])  

def increment(cycle, i, j):
    return dist[cycle[i]-1, j-1] + dist[j-1, cycle[i+1]-1] - dist[cycle[i]-1, cycle[i+1]-1]

while W:
    best_increase = INF
    best_vertex = None
    best_pos = None
    
    for j in W:
        for i in range(len(cycle)-1):
            inc = increment(cycle, i, j)
            if inc < best_increase:
                best_increase = inc
                best_vertex = j
                best_pos = i+1
    cycle.insert(best_pos, best_vertex)
    W.remove(best_vertex)

total_cost = 0
for i in range(len(cycle)-1):
    total_cost += dist[cycle[i]-1, cycle[i+1]-1]

print("Ciclo Hamiltoniano aproximado:", cycle)
print("Costo total del viaje:", total_cost)


Ciclo Hamiltoniano aproximado: [1, 2, 4, 3, 5, 6, 7, 1]
Costo total del viaje: 65.0
